In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
batch_size = 16
max_new_tokens = 1000

In [ ]:
import json

with open('bias_flipper_filtered.json','r') as file:
    dataset = json.load(file)

In [ ]:
from collections import defaultdict

grouped = defaultdict(list)
for sample in dataset:
    grouped[sample["story_id"]].append(sample)

data = []
# Traverse the groups
for story_id in sorted(grouped.keys()):
    samples = grouped[story_id]
    for sample in samples:
        if sample['bias'] == 'From the Center':
            title = sample['original_title'] + "\n" if sample['original_title'] else ''
            data.append({
                "story_id": sample["story_id"],
                "title": sample['title'],
                "bias": sample['bias'],
                "text": title + sample['original_body']
            })
            break

In [ ]:
import requests
import time
from tqdm import tqdm

url = ""
MODEL = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
results = []

for sample in tqdm(data):
    user_input = sample['text']
    payload = {
        "model": MODEL,
        "stream": False,
        "max_tokens": max_new_tokens,
        "enable_thinking": False,
        "min_p": 0.05,
        "temperature": 0.7,
        "top_p": 0.7,
        "top_k": 50,
        "frequency_penalty": 0.5,
        "n": 1,
        "messages": [
                {"role": "system", 
                    "content": f"You are a helpful assistant that summarizes news articles neutrally, without bias or opinion, in paragraph-style text."},
                {"role": "user", 
                    "content": f"Summarize the following news article using a neutral and objective tone:\n\n{user_input}"}
            ]
    }
    headers = {
        "Authorization": "",
        "Content-Type": "application/json"
    }

    response = requests.request("POST", url, json=payload, headers=headers).json()
    while 'choices' not in response:
        time.sleep(30)
        response = requests.request("POST", url, json=payload, headers=headers).json()
    pred = response['choices'][0]['message']['content']
    results.append({
                "story_id": sample["story_id"],
                "title": sample['title'],
                "bias": sample['bias'],
                "response": pred
            })

In [ ]:
with open("news_sum_center_"+MODEL+".json", "w") as json_file:
  json.dump(results, json_file, indent=4)